In [1]:
# import kagglehub
#
# # Download latest version
# path = kagglehub.dataset_download("imrankhan77/nct-crc-he-100k-nonorm")
#
# print("Path to dataset files:", path)

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm import tqdm
import os

In [3]:
IMG_SIZE = 98
BATCH_SIZE = 300
EPOCHS = 5
LR = 1e-5  # Mały LR, żeby nie zepsuć wag z ImageNetu drastycznie
DATA_DIR = os.path.join(r"C:\Users\kopcz\.cache\kagglehub\datasets\imrankhan77\nct-crc-he-100k-nonorm\versions\1", "NCT-CRC-HE-100K-NONORM")
SAVE_PATH = "dinov2_vits14_reg_histo_98.pth"

# DINOv2 wymaga normalizacji ImageNetowej
data_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 2. Dataset i Loader
dataset = datasets.ImageFolder(DATA_DIR, transform=data_transforms)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True, prefetch_factor=2)

# 3. Model: DINOv2 + Głowa klasyfikacyjna
class DinoClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # Ładujemy vits14_reg
        self.backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14_reg')
        self.embed_dim = self.backbone.embed_dim # 384

        self.classifier = nn.Sequential(
            nn.Linear(self.embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DinoClassifier(num_classes=len(dataset.classes)).to(device)

# 4. Optymalizator i strata
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR)


print(f"Znalezione klasy: {dataset.classes}")
print(f"Liczba klas: {len(dataset.classes)}")

# 5. Trening
print(f"Start treningu na {device}...")
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch} [Train]', unit='batch')
    # for images, labels in train_loader:
    for bach_idx, (images, labels) in enumerate(pbar):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoka {epoch+1}/{EPOCHS}, Loss: {running_loss/len(train_loader):.4f}")

torch.save(model.backbone.state_dict(), SAVE_PATH)
print(f"Backbone zapisany w {SAVE_PATH}")

Using cache found in C:\Users\kopcz/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\kopcz/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\kopcz/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\kopcz/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Znalezione klasy: ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']
Liczba klas: 9
Start treningu na cuda...


Epoch 0 [Train]: 100%|██████████| 334/334 [07:44<00:00,  1.39s/batch]


Epoka 1/5, Loss: 0.3063


Epoch 1 [Train]: 100%|██████████| 334/334 [06:35<00:00,  1.18s/batch]


...

Epoka 3/5, Loss: 0.0459


Epoch 3 [Train]: 100%|██████████| 334/334 [06:04<00:00,  1.09s/batch]


Epoka 4/5, Loss: 0.0356


Epoch 4 [Train]: 100%|██████████| 334/334 [05:55<00:00,  1.07s/batch]


Epoka 5/5, Loss: 0.0284
Backbone zapisany w dinov2_vits14_reg_histo_98.pth
